In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
%matplotlib inline

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import(accuracy_score,classification_report,confusion_matrix)

from imblearn.over_sampling import SMOTE

In [3]:
import warnings 
warnings.filterwarnings("ignore")

In [4]:
df=pd.read_csv("wine.csv")
df.head()

,wine,Alcohol,Malic acid,Ash,Alcalinity of ash,Magnesium,Total phenols,Flavanoids,Nonflavanoid phenols,Proanthocyanins,Color intensity,Hue,OD280/OD315 of diluted wines,Proline
0,1,14.23,1.71,2.43,15.6,127,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065
1,1,13.20,1.78,2.14,11.2,100,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050
2,1,13.16,2.36,2.67,18.6,101,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185
3,1,14.37,1.95,2.50,16.8,113,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480
4,1,13.24,2.59,2.87,21.0,118,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 178 entries, 0 to 177
Data columns (total 14 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   wine                          178 non-null    int64  
 1   Alcohol                       178 non-null    float64
 2   Malic acid                    178 non-null    float64
 3   Ash                           178 non-null    float64
 4   Alcalinity of ash             178 non-null    float64
 5   Magnesium                     178 non-null    int64  
 6   Total phenols                 178 non-null    float64
 7   Flavanoids                    178 non-null    float64
 8   Nonflavanoid phenols          178 non-null    float64
 9   Proanthocyanins               178 non-null    float64
 10  Color intensity               178 non-null    float64
 11  Hue                           178 non-null    float64
 12  OD280/OD315 of diluted wines  178 non-null    float64
 13  Proli

In [8]:
df.rename(columns={

    'Alcohol':'alcohol',
    'Malic acid':'malic_acid',
    'Ash':'ash',
    'Alcalinity of ash':'alcalinity_of_ash',
    'Magnesium':'magnesium',
    'Total phenols':'tot_phenols',
    'Flavanoids':'flavanoids',
    'Nonflavanoid phenols':'nonflavanoid_phenols',
    'Proanthocyanins':'proanthocyanins',
    'Color intensity':'color_intensity',
    'Hue':'hue',
    'OD280/OD315 of diluted wines':'diluted_wines',
    'Proline':'proline'

}, inplace=True)

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 178 entries, 0 to 177
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   wine                  178 non-null    int64  
 1   alcohol               178 non-null    float64
 2   malic_acid            178 non-null    float64
 3   ash                   178 non-null    float64
 4   alcalinity_of_ash     178 non-null    float64
 5   magnesium             178 non-null    int64  
 6   tot_phenols           178 non-null    float64
 7   flavanoids            178 non-null    float64
 8   nonflavanoid_phenols  178 non-null    float64
 9   proanthocyanins       178 non-null    float64
 10  color_intensity       178 non-null    float64
 11  hue                   178 non-null    float64
 12  diluted_wines         178 non-null    float64
 13  proline               178 non-null    int64  
dtypes: float64(11), int64(3)
memory usage: 19.6 KB


In [10]:
df.isnull().sum()

wine                    0
alcohol                 0
malic_acid              0
ash                     0
alcalinity_of_ash       0
magnesium               0
tot_phenols             0
flavanoids              0
nonflavanoid_phenols    0
proanthocyanins         0
color_intensity         0
hue                     0
diluted_wines           0
proline                 0
dtype: int64

In [11]:
df.duplicated().sum()

np.int64(0)

In [12]:
X=df.iloc[:,1:]
y=df.wine

In [14]:
# Detect Outliers using IQR

import pandas as pd

# Select numerical columns
num_cols = df.select_dtypes(include=['int64', 'float64']).columns

for col in num_cols:

    # Q1 and Q3
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    # IQR
    IQR = Q3 - Q1

    # Lower and Upper limits
    lower_limit = Q1 - 1.5 * IQR
    upper_limit = Q3 + 1.5 * IQR

    # Outliers
    outliers = df[(df[col] < lower_limit) | 
                    (df[col] > upper_limit)]

    print(f"{col} : {outliers.shape[0]} outliers")

wine : 0 outliers
alcohol : 0 outliers
malic_acid : 3 outliers
ash : 3 outliers
alcalinity_of_ash : 4 outliers
magnesium : 4 outliers
tot_phenols : 0 outliers
flavanoids : 0 outliers
nonflavanoid_phenols : 0 outliers
proanthocyanins : 2 outliers
color_intensity : 4 outliers
hue : 1 outliers
diluted_wines : 0 outliers
proline : 0 outliers


In [15]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=42)

In [24]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import StandardScaler
import pandas as pd
num_col = X_train.select_dtypes(include=['int64', 'float64']).columns

ct = ColumnTransformer(transformers=[
    ("sc", StandardScaler(), num_col)
])

# Train data transform
X_train_transform = ct.fit_transform(X_train)
X_train_transform = pd.DataFrame(X_train_transform)

print(X_train_transform)

# Test data transform
X_test_transform = ct.transform(X_test)
X_test_transform = pd.DataFrame(X_test_transform)

print(X_test_transform)

           0         1         2             3         4         5         6   \
0    0.628447  1.081206 -0.652127  2.820325e-17 -0.841477 -1.003358 -1.517062   
1   -0.540882 -0.612994 -1.427534  2.881798e-01 -1.037487 -0.112585 -0.086751   
2   -0.755657 -1.287031 -1.538306 -1.354445e+00  2.294697 -0.573329 -0.156280   
3    0.377877 -0.694972  1.747940 -1.152719e+00  0.595936  0.501741  0.668135   
4   -0.803385  0.388952 -0.541355 -4.322697e-01 -0.841477  0.271369  0.241029   
..        ...       ...       ...           ...       ...       ...       ...   
119  1.069929 -0.813384  1.120230  1.584989e+00 -0.972150  1.039276  0.846924   
120 -0.851113 -0.612994 -0.910596 -1.440899e-01 -1.364172 -0.957283  0.022509   
121  1.690390 -0.485474  0.049431 -2.161348e+00  0.073241  1.576811  1.621676   
122 -0.326107 -0.795166 -0.393659  3.458157e-01 -1.364172 -1.371953 -0.543656   
123 -0.743725  0.042825  0.344824  4.322697e-01 -0.188107  0.440308  0.101971   

           7         8     

In [25]:
from collections import Counter

In [26]:
print(Counter(y_train))

Counter({2: 50, 1: 40, 3: 34})


In [29]:
sm=SMOTE()
X_smote,y_smote=sm.fit_resample(X_train_transform,y_train)
print(Counter(y_smote))

Counter({3: 50, 2: 50, 1: 50})


# Bagging 

In [32]:
from sklearn.ensemble import BaggingClassifier
from sklearn.linear_model import LogisticRegression

# Create Logistic Regression object
LR = LogisticRegression()

# Create Bagging Classifier
model_bagg = BaggingClassifier(
    estimator=LR,
    n_estimators=20
)

# Train model
model_bagg.fit(X_smote, y_smote)

# Predictions
y_hat_bagg = model_bagg.predict(X_test_transform)

In [33]:
# Training Score
train_score = model_bagg.score(X_smote, y_smote)

# Testing Score
test_score = model_bagg.score(X_test_transform, y_test)

# Display scores
print("Training Score:", train_score)
print("Testing Score:", test_score)

Training Score: 1.0
Testing Score: 0.9814814814814815


In [37]:
from sklearn.ensemble import RandomForestClassifier

# Create model
rf = RandomForestClassifier()

# Train model
rf.fit(X_smote, y_smote)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [39]:
# Training Score
train_score = model_bagg.score(X_smote, y_smote)

# Testing Score
test_score = model_bagg.score(X_test_transform, y_test)

# Display scores
print("Training Score:", train_score)
print("Testing Score:", test_score)

Training Score: 1.0
Testing Score: 0.9814814814814815
